# Supplemental Material figures

Produces the SI's numerical figures from `extremal_mil.si_experiment`:

```
OMP_NUM_THREADS=1 python -m extremal_mil.si_experiment
```

1. the gradient field swept over $u$ at several $\alpha$ and, in a second row,
   at several initial $\lambda_0$,
2. $\dot\lambda$ against $\lambda_0$ at fixed $u$, one panel in R and one in N.

Both use $\ell=100$: the asymptotics are a statement about $d$ at fixed
$\alpha$, so the convergence should be visible at any fixed $\ell$.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from extremal_mil import plotting, theory
from extremal_mil import si_experiment as si

plt.rcParams.update(plotting.RCPARAMS)
%config InlineBackend.figure_format = 'svg'

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COMPONENTS = [("lam", r"$\dot\lambda$"), ("u", r"$\dot u$"), ("rho", r"$\dot\rho$")]
print(f"alpha values {si.ALPHA_VALUES}, lam0 values {si.LAM0_VALUES}, ell={si.ELL}")

## 1. The gradient field at several $\alpha$ (top row) and initial $\lambda_0$ (bottom row)

In [ ]:
def curve_row(fig, cells, label, title, u_star_of):
    # One row of (dlam, du, drho) against u, one colored curve per cell, drawn
    # into ``fig`` (a subfigure).
    colors, _ = plotting.run_colors(np.arange(len(cells)))
    _, axes = plotting.new_row(len(COMPONENTS), fig=fig)
    for ax, (key, ylab) in zip(axes[0], COMPONENTS):
        plotting.zero_line(ax)
        for x, c in zip(cells, colors):
            ax.plot(x["u"], x[f"emp_{key}"], color=c, lw=1.2, alpha=0.9)
            ax.plot(x["u"], x[f"th_{key}"], color=c, lw=1.0, ls="--", alpha=0.85)
            ax.axvline(u_star_of(x), color=c, lw=0.8, ls=":", alpha=0.8)
        ax.set_ylabel(ylab)
    plotting.finish_row(
        fig, axes, title, "$u$",
        handles=plotting.series_handles(cells, colors, label)
        + [Line2D([], [], color="0.3", ls="--", lw=1.0, label="theory"),
           Line2D([], [], color="0.3", ls=":", lw=0.8, label=r"$u^\star$")])


ELL_FIG = 100

# Top row: one curve per alpha, at the largest d available for each (n = 1808
# at all three, so the bag size is held and only alpha moves).
cells_alpha = [si.load_u_curves(a, si.LAM0, si.D_VALUES_ALPHA[a][-1], ell=ELL_FIG)
               for a in si.ALPHA_VALUES]
for x in cells_alpha:
    print(f"alpha={float(x['alpha']):.3f}  d={int(x['d'])}  n={int(x['n'])}  "
          f"lambda*={float(x['lam_star']):.4f}  u*={float(x['u_star']):.4f}")

# Bottom row: lambda* = sqrt(2 alpha) = 0.316, so lam0 = 0.05 and 0.2 start in
# H and 0.5 in L.
cells_lam0 = [si.load_u_curves(si.ALPHA, lam0, si.D_VALUES[-1], ell=ELL_FIG)
              for lam0 in si.LAM0_VALUES]
for x in cells_lam0:
    ph = "H" if float(x["lam0"]) < float(x["lam_star"]) else "L"
    print(f"lam0={float(x['lam0']):.2f} ({ph})  u*={float(x['u_star']):.4f}")

fig = plt.figure(figsize=(plotting.TEXT_W, 4.8), layout="constrained")
top, bottom = fig.subfigures(2, 1, hspace=0.04)
curve_row(top, cells_alpha,
          lambda x: rf"$\alpha={float(x['alpha']):g}$ ($d={int(x['d'])}$)",
          rf"(a) $\lambda_0={si.LAM0}$",
          lambda x: float(x["u_star"]))
curve_row(bottom, cells_lam0,
          lambda x: rf"$\lambda_0={float(x['lam0']):g}$",
          rf"(b) $d={si.D_VALUES[-1]}$",
          lambda x: float(x["u_star"]))
fig.savefig(FIG_DIR / "si_alpha_lam0.pdf", bbox_inches="tight")
plt.show()

## 2. $\dot\lambda$ against $\lambda_0$ at fixed $u$

In [ ]:
# The transposed sweep. u is held at a value that keeps the point in one phase
# across the whole lambda grid: 0.1 stays in N, 0.7 stays in R.
PANELS = [(si.U_NON_RETRIEVAL, "N (non-retrieval)"), (si.U_RETRIEVAL, "R (retrieval)")]

d_have = si.D_VALUES_LAM
colors, _ = plotting.run_colors(np.arange(len(d_have)))
fig, axes = plotting.new_row(len(PANELS))

for ax, (u0, phase) in zip(axes[0], PANELS):
    cells = [si.load_lam_curves(u0, d, ell=ELL_FIG) for d in d_have]
    plotting.zero_line(ax)
    ax.axvline(float(cells[0]["lam_star"]), zorder=1, **plotting.HL_STYLE)
    for x, c in zip(cells, colors):
        ax.plot(x["lam"], x["emp_lam"], color=c, lw=1.2, alpha=0.9)
    ax.plot(cells[0]["lam"], cells[0]["th_lam"], zorder=3, **plotting.THEORY_STYLE)
    ax.set_ylabel(r"$\dot\lambda$")
    ax.set_title(rf"$u={u0}$", pad=4)

plotting.finish_row(
    fig, axes, None, r"$\lambda_0$",
    handles=plotting.series_handles(
        [si.load_lam_curves(si.U_RETRIEVAL, d, ell=ELL_FIG) for d in d_have], colors,
        lambda x: rf"$d={int(x['d'])}$")
    + [plotting.theory_handle(), Line2D([], [], label=r"$\lambda^\star$",
                                        **plotting.HL_STYLE)])
fig.savefig(FIG_DIR / "si_lam_sweep.pdf", bbox_inches="tight")
plt.show()